# AI Agent Integration with HPC Slurm Jobs

In this tutorial, you will use an **AI agent framework** to instantiate an AI agent to help you write code and generate Slurm job scripts for submitting code as jobs to an HPC system scheduler.

This agent framework is built around the `Agent` class (see `./TACC_exAI/agent.py`) and supports multiple **actions** such as:
- Creating and running multi-step plans.
- Summarizing context and replying to the user.
- Generating code snippets.
- Writing Slurm job scripts for HPC clusters.
- Optional runtime tracing with Arize Phoenix for observability.

Just like in previous tutorials, the underlying language model runs on a local **Ollama** server using an OpenAI-compatible API, so everything stays on your machine while still using an LLM backend.

## Add **TACC_exAI** framework path
This allows us to import it as a python module

In [ ]:
# Import TACC_exAI framework folder
import sys
import os

# Resolve "../TACC_exAI" relative to the current working directory
new_path = os.path.abspath(os.path.join(os.getcwd(), "..","..", ".."))
if new_path not in sys.path:
    sys.path.insert(0, new_path)  # or .append(new_path)

## Planning Style Agents

In this tutorial we will put together a planning agent that will first generate a plan for a series of actions that it will perform in sequence. This allows the agent to coordinate long range dependencies between actions, enabling it to tackle longer tasks autonomously.

In [ ]:
# Launch Ollama serve in background, pipe output to log file
import subprocess
import os
import signal
import time
import atexit

# Create log file
log_file = "ollama_server.log"

# Kill any existing ollama processes
subprocess.run(["pkill", "ollama"], capture_output=True)
time.sleep(2)

#os.environ['OLLAMA_MODELS']='./ollama_models'
os.environ['OLLAMA_MODELS']
os.environ['OLLAMA_DEBUG']='1'
os.environ['OLLAMA_LOG_LEVEL']='debug'



# Start ollama serve in background
print("🚀 Starting Ollama server...")
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open(log_file, "w"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setpgrp
)
print(f"📄 Server logs: {log_file}")
print(f"📍 API endpoint: http://localhost:11434")

# Wait for server to start
print("⏳ Waiting 5 seconds for server startup...")
time.sleep(5)

# Store process PID globally for cleanup (avoid %store magic)
if 'OLLAMA_PROCESS' not in globals():
    OLLAMA_PROCESS = process.pid
    print(f"✅ Ollama server ready! PID: {OLLAMA_PROCESS}")
else:
    print("✅ Ollama server already running!")

# Register cleanup function
def cleanup_ollama():
    try:
        if 'OLLAMA_PROCESS' in globals():
            os.killpg(os.getpgid(OLLAMA_PROCESS), signal.SIGTERM)
            print("🛑 Ollama server stopped")
    except:
        pass

atexit.register(cleanup_ollama)

## Configuring a Long Context Model on the Local Ollama Backend

The `Agent` class in `agent.py` is designed to be flexible: you can run it from the command line or import and use it as a Python class. In this tutorial, you will work with it directly as a class object inside this notebook.

Below is code that will configure our ollama instance to have a 20,000 token context limit for the qwen3-code:30b model. We will point our agent to this new long context model `qwen3-coder:30b-20k`

# Create the extended model in Ollama
This code is how we are able to customize a model available from Ollamas model repository
It creates a Modelfile which specifies a BASE MODEL (one that is readily available in Ollamas repository)

```
import subprocess
import os

# Create the Modelfile in the current directory
#modelfile_content = '''FROM qwen3-coder:30b

#PARAMETER num_ctx 20000

#TEMPLATE """{{- if .System }}<|im_start|>system
#{{ .System }}<|im_end|>
#{{- end }}{{- if .Prompt }}<|im_start|>user
#{{ .Prompt }}<|im_end|>
#{{- end }}<|im_start|>assistant
#{{ .Response }}<|im_end|>"""'''


modelfile_content = '''FROM qwen3-coder-next

TEMPLATE """{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{- end }}{{- if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{- end }}<|im_start|>assistant
{{ .Response }}<|im_end|>"""'''


with open("Modelfile", "w") as f:
    f.write(modelfile_content)

# Run the ollama create command
model_name = "qwen3-coder-next"
print(f"Creating model {model_name} ...")

try:
    result = subprocess.run(
        ["ollama", "create", model_name, "-f", "Modelfile"],
        capture_output=True,
        text=True,
        check=True
    )
    print("✅ Model created successfully!")
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print("❌ Error creating model:")
    print(e.stderr)
finally:
    # Optionally clean up Modelfile
    if os.path.exists("Modelfile"):
        os.remove("Modelfile")
```

## 1. Single-Tool Chatbot

We will start with the **simplest possible configuration** of this agent: give it **one tool** and let it execute that tool once, so it behaves like a basic chatbot.

Key choices for this first example:
- Use an **isolated session** so the agent does *not* load or reuse any previous conversation history.
- Disable session saving with `no_save=True` so no history is written to disk.
- Configure `self.actions` to contain only `SummarizeAndReplyAction`, so the agent simply summarizes the user input and responds.

Conceptually, you can think of it as: *"The agent receives a message, runs a single Summarize-and-Reply step, and returns a friendly answer."*

In [ ]:
import sys
import os

# Add the agent framework code to our system path so we can import it
#notebook_dir = os.getcwd()
#notebook_dir = os.path.join(notebook_dir, "TACC_exAI")
#if notebook_dir not in sys.path:
#    sys.path.insert(0, notebook_dir)

# import agent framework
from TACC_exAI.agent import Agent
from TACC_exAI.actions.summarize_and_reply import SummarizeAndReplyAction

# Instantiate the agent as a single-turn chatbot:
agent = Agent(
    experiment=True,           # run one non-interactive cycle
    experiment_prompt=None,    # we'll set the prompt manually below
    force_ollama=True,         # use our local ollama backend as our llm inferencing provider
    default_action_model_name="qwen3-coder-next",  # set the model name to use
    isolated_session=True,     # do not load logs of other chats
    no_save=True,              # do not save current conversation to chat history logs
    display_mode="light",      # configure console display color pallete
    mode="dev",                # verbose logging outputs including prompts
)

# The agent will run each action listed in this array in series when agent.run() is called
# Note: we need to give our action a reference to the agent so it can read/write to agent 
# runtime variables
agent.actions = [SummarizeAndReplyAction(agent=agent)]

# Give the agent a simple prompt: ask for a dad joke.
agent.experiment_prompt = (
    "My SLURM job vanished from the queue. "
    "Give me a short story (of about 150 words) of Sherlock Holmes and Dr. Watson tracing the whereabouts of this missing job. "
    "Make Moriarty and Mycroft Holmes cruicial in the plot. "
    "End the passage with a joke about the reliability of software developers. "
    "Separate all sentences by two newlines. "
)

# Run the agent once and capture the reply.
response = agent.run()
print("\nAgent response:")
print(response)

In this configuration:
- `agent.actions` contains only `SummarizeAndReplyAction`, so the agent’s pipeline is a single step: summarize the input and respond to the user.
- Because `experiment=True`, `agent.run()` processes a single prompt (stored in `agent.experiment_prompt`) and then returns the final assistant reply instead of entering an interactive loop.
- Using `isolated_session=True` and `no_save=True` prevents any previous or future sessions from influencing this run.

The effect is a simple, well-contained chatbot that behaves similarly to the structured joke generator you built in Tutorial 1, but now implemented on top of a general-purpose agent framework.

## 2. Planning and Tool Use Agent

Now we will enable more of the agent framework’s features. Instead of executing a single fixed action, the agent will:

1. **Create a plan** using structured generation that describes a sequence of actions to accomplish the user’s request.
2. **Run the plan**, invoking the appropriate tools (actions) in order. Revises plan on Action Failures

We will configure:
- `self.actions` (the pipeline) to include `CreatePlanAction` followed by `RunPlanAction`.
- `self.available_actions` (the tool set) to include:
  - `SummarizeAndReplyAction`: summarize the messages in the agent's context and compose a new message to the user.
  - `GenerateCodeAction`: write Python code to solve the problem, execute it, and report the execution outputs.

In [ ]:
from TACC_exAI.actions.create_plan import CreatePlanAction
from TACC_exAI.actions.run_plan import RunPlanAction
from TACC_exAI.actions.generate_code import GenerateCodeAction

# Instantiate another agent configured for planning + tool use.
plan_agent = Agent(
    experiment=True,           # run one non-interactive cycle
    experiment_prompt=None,    # we'll set the prompt manually below
    force_ollama=True,         # use our local ollama backend as our llm inferencing provider
    default_action_model_name="qwen3-coder-next",  # set the model name to use
    isolated_session=True,     # do not load logs of other chats
    no_save=True,              # do not save current conversation to chat history logs
    display_mode="light",      # configure console display color pallete
    mode="dev",                # verbose logging outputs including prompts
    use_apptainer=True
)

# Configure the pipeline actions: first create a plan, then run it.
plan_agent.actions = [
    CreatePlanAction(agent=plan_agent),#, tracer=None),
    RunPlanAction(agent=plan_agent),#, tracer=None),
]

# Restrict the available tools the plan can choose from.
plan_agent.available_actions = [
    SummarizeAndReplyAction(),
    GenerateCodeAction(),
]

# User task: write and reason about analysis code.
plan_agent.experiment_prompt = (
    "I want you to first write code that"
    " reports the speed of searching a random 1000 object test array using two different methods."
    " After writing the code, message me with a final report summarizing the code output benchmark"
    " results and a theory for why one method is faster."
    
)

plan_response = plan_agent.run()

## Exercise

Have the agent create code to monitor the number of pending jobs in a SLURM queue.
(Copy over the code from the previous cell and modify the prompt with appropriate instructions.)

### Solution

```
from actions.create_plan import CreatePlanAction
from actions.run_plan import RunPlanAction
from actions.generate_code import GenerateCodeAction

# Instantiate another agent configured for planning + tool use.
plan_agent = Agent(
    experiment=True,           # run one non-interactive cycle
    experiment_prompt=None,    # we'll set the prompt manually below
    force_ollama=True,         # use our local ollama backend as our llm inferencing provider
    default_action_model_name="qwen3-coder-next",  # set the model name to use
    isolated_session=True,     # do not load logs of other chats
    no_save=True,              # do not save current conversation to chat history logs
    display_mode="light",      # configure console display color pallete
    mode="dev",                # verbose logging outputs including prompts
    use_apptainer=True
)

# Configure the pipeline actions: first create a plan, then run it.
plan_agent.actions = [
    CreatePlanAction(agent=plan_agent),#, tracer=None),
    RunPlanAction(agent=plan_agent),#, tracer=None),
]

# Restrict the available tools the plan can choose from.
plan_agent.available_actions = [
    SummarizeAndReplyAction(),
    GenerateCodeAction(),
]

# User task: write and reason about analysis code.
plan_agent.experiment_prompt = (
    "Create a python script to continuously monitor the number of pending jobs in a specific slurm queue."
    "Ensure a polling frequency of 1 request per 30 seconds to ensure not overloading the slurm control daemon."
    "The entire code should be guarded by a flag that skips its execution by default."
    
)

plan_response = plan_agent.run()
```

## How the planning flow works

With this configuration, a typical execution sequence looks like:

1. **CreatePlanAction** inspects the prompt and generates a structured plan: for example, steps like *"call GenerateCodeAction with these instructions"* followed by *"call SummarizeAndReplyAction with the results"*.
2. **RunPlanAction** executes each step in order, handing off context (such as generated code or intermediate outputs) between actions.
3. **GenerateCodeAction** produces the requested Python code, executes it in an isolated environment, revises errors, and reports the console outputs back to the agent.
4. **SummarizeAndReplyAction** summarizes the code outputs and responds with a user-friendly summary as the final answer.

In later steps of this tutorial, you will extend this pattern by enabling additional actions such as **GenerateSlurmScriptAction** so that the agent can not only write analysis code but also author complete Slurm job scripts that you can submit on your HPC system.

You now have an end-to-end workflow where an AI agent plans, generates, and refines Slurm job scripts for your HPC workloads. In the next tutorial, you can extend this pattern to more complex agents that analyze job outputs, adapt parameters, or orchestrate multi-stage HPC pipelines.

## 3. Generating Slurm scripts

This section demonstrates how the AI agent can both author and operationalize compute tasks on an HPC system. Below we will configure the agent to both write and test code and generate an HPC Slurm job script that runs that code on multiple nodes with GPU resources.

The agent will:

1) Write and test a short Python script that records the node hostname and hardware logging output from the node it runs on.

2) Generate a Slurm job script that would launch this Python code on multiple nodes, one task per node, and then aggregate the results.



In [ ]:
import sys
import os

# Add the agent framework code to our system path so we can import it
notebook_dir = os.getcwd()
notebook_dir = os.path.join(notebook_dir, "TACC_exAI")
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

# import agent framework
from TACC_exAI.agent import Agent
from TACC_exAI.actions.summarize_and_reply import SummarizeAndReplyAction
from TACC_exAI.actions.create_plan import CreatePlanAction
from TACC_exAI.actions.run_plan import RunPlanAction
from TACC_exAI.actions.generate_code import GenerateCodeAction

from TACC_exAI.actions.generate_slurm_script import GenerateSlurmScriptAction


# Initialize the agent
agent = Agent(
    experiment=True,
    experiment_prompt=None,
    #force_ollama=False,
     force_ollama=True,
    default_action_model_name="qwen3-coder-next",
   # default_action_model_name="llama3.2",  # long context model
    isolated_session=True,
    no_save=True,
    display_mode="light",
    mode="dev",
    use_apptainer=True
)

# Configure toolset: enable code generation and slurm generation
agent.available_actions = [
    GenerateCodeAction(agent=agent),
    GenerateSlurmScriptAction(agent=agent),
    SummarizeAndReplyAction(agent=agent),
]

# Configure pipeline: plan creation and plan execution
agent.actions = [
    CreatePlanAction(agent=agent),
    RunPlanAction(agent=agent),
]

# Set the experiment prompt

agent.experiment_prompt = (
    "First, write a Python script that saves the hostname and the output "
    "of some general system statistics using psutils to a local file named `node_info_<hostname>.txt` "
    " and prints that information to the console."
    "Implement a flag around this code that skips its execution by default with a comment that the user should toggle the flag in order to run it."
    "Then, create a Slurm job script that runs this Python code across 2 nodes "
    "(1 task per node), requesting GPUs appropriately. "
    "After all tasks finish, combine all the generated output files "
    "into one file named `combined_node_info.txt`."
)

# Run the agent and capture output
response = agent.run()
print("\nAgent response:")
print(response)


## Slurm Script Chaining for Multi-Job Workflows

In this section, we task our agent to handle a more complex, multi‑stage workflow that combines code authoring, Slurm script generation, and iterative job submission.  

We will instruct the agent to:

1. **Write a Python script** that computes digits of π to high precision using the Chudnovsky algorithm (or similar).  
   - The script will store results in a file, e.g. `pi_digits.txt`.  
   - On each run, it will detect how many digits are already present and extend the file by another 10,000 digits.  
   - After completion, it will report how many digits are now stored.

2. **Create a Slurm job submission script** to run this computation efficiently on an HPC system.  
   - The job requests reasonable CPU, memory, and walltime resources.  
   - All standard output and error streams will be captured in log files (e.g., `pi_job.out`, `pi_job.err`).

3. **Write a Python “launch” script** that uses `sbatch` to queue jobs iteratively.  
   - Each job will sit in the queue, but begin only after the previous one finishes (using Slurm’s dependency feature).  
   - This allows automated, chained computation expansions (e.g., 10,000 → 20,000 → 30,000 digits) that are common in model training and simulation workflows.  
   - Because we plan to test and launch this script manually, we ask the agent to bypass its normal code execution test. To achieve this, we wrap the runnable logic in a **guarded flag block**—defaulting to `True`—which users can later toggle to enable actual execution.

The code cell below initializes the agent, configures its available toolset and planning pipeline, and then defines a detailed prompt specifying the desired behavior. Once executed, the agent will design and output all three scripts—each adapted for HPC use via Slurm.

> **Tip:** The final “launch” script demonstrates how to **chain Slurm jobs automatically** to build cumulative results without manual re‑submission—a powerful automation pattern for iterative HPC workloads.

In [ ]:
import sys
import os

# Add the agent framework code to our system path so we can import it
notebook_dir = os.getcwd()
notebook_dir = os.path.join(notebook_dir, "TACC_exAI")
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

# import agent framework
from TACC_exAI.agent import Agent
from TACC_exAI.actions.summarize_and_reply import SummarizeAndReplyAction
from TACC_exAI.actions.create_plan import CreatePlanAction
from TACC_exAI.actions.run_plan import RunPlanAction
from TACC_exAI.actions.generate_code import GenerateCodeAction

from TACC_exAI.actions.generate_slurm_script import GenerateSlurmScriptAction


# Initialize the agent
agent = Agent(
    experiment=True,
    experiment_prompt=None,
    #force_ollama=False,
    force_ollama=True,
    #default_action_model_name="DeepSeek-V3-0324",  # long context model
    default_action_model_name="qwen3-coder-next",  # long context model
    isolated_session=True,
    no_save=True,
    display_mode="light",
    mode="dev",
    use_apptainer=True
    
)

# Configure toolset: enable code generation and slurm generation
agent.available_actions = [
    GenerateCodeAction(agent=agent),
    GenerateSlurmScriptAction(agent=agent),
    SummarizeAndReplyAction(agent=agent),
]

# Configure pipeline: plan creation and plan execution
agent.actions = [
    CreatePlanAction(agent=agent),
    RunPlanAction(agent=agent),
]

# Set the experiment prompt

agent.experiment_prompt = (
    f"Write three scripts:\n"
    f"1. **A Python script** that computes pi to 10,000 digits (using a high‑precision method such as "
    f"Chudnovsky), reads any previously computed digits from a file (e.g., `pi_digits.txt`) if it exists,"
    f" extends the total by 10,000 digits beyond what is already stored, saves the updated digits back "
    f"to the file, and prints how many digits are now stored before exiting."
    f" Place this code under a flag that defaults to skipping its execution.\n"
    f"2. **A Slurm script** that submits this Python script as a job, requests reasonable CPU, memory, "
    f"and walltime, and writes stdout and stderr to log files (e.g., `pi_job.out` and `pi_job.err`), "
    f" exiting when the Python script finishes.\n"
    f"3. **A Python launch script** that submits the Slurm script **iteratively** using sbatch to queue "
    f"successive runs so that each job waits in the slurm queue but starts only after the previous job "
    f" finishes, adding 10,000 more digits of pi with each run (from 10,000 → 20,000 → 30,000), without "
    f"checking the file contents; assume each run simply appends 10,000 more digits on top of the prior "
    f"result. Use sbatch features to accomplish the slurm script chaining.\n"
    f"Note: Write the final script with a flag default to true that skips the whole script and tells "
    f"the user to edit the script to flip it to false. Within the skipped section, give the script "
    f"a CLI to ingest the path to the slurm and python scripts, I'll run it later."
)
# Run the agent and capture output
response = agent.run()
print("\nAgent response:")
print(response)

# Take home exercises

1. Examine and Try manually running the scripts generated in these last 2 examples to verify their functionality
    -  **Note:** There will be the following modifications required:
        -  For the **Python scripts** :
            -  You will need to toggle the execution flag variable if present.
            -  You should run the python scripts on a compute node and **NOT** on the login node.
            -  Get a compute node session on command line using the "idev" command **OR**
            -  Within the notebook session, navigate to "File" -> "New" -> "Terminal" and go the the new terminal tab
        -  For the **SLURM scripts** *(**AND** for any Python code that is invoking a slurm command):  
            -  These cannot be run on a compute node and should be run on a login node
                -  Obtain an ssh session to a Vista login node via a command line terminal
                -  Again for the python code, you will need to toggle the execution flag variable
                -  For SLURM scripts, you will need to add further information in order to get it to work:   
                    -  Your Project Account (-A " " parameter to sbatch)  
                    -  Your Reservation Details (--reservation " " parameter to sbatch)
                        - On a login node commandline session run `scontrol show reservations` to see the name of the reservation you are listed on.



2. Prompt Alteration Exercise:
   Alter the prompt in the last example to create a single job with multiple jobsteps, each running a different instance of the pi calculation.  
   Each jobstep should output to a different file. The final jobstep should parse all the output files of the pi calculation and calulate the average of all values
   and output to the console.

   
   